# Phase C — MODEL-SEED CI (CPU, Kaggle)

Loads `phase_C_preds_seed{42,100,200}.pkl` and runs the CORRECTED conformal
benchmark (lambda=3, gamma_max=0.15, PB-JCI Online, temporal_drift) for
3 model x 5 cal = 15 runs. Aggregates mean+/-std -> Table 4c.

**Attach dataset:** `hipinhththu/sam3-q1-multiseed-ckpts` (holds the 3 pkls).
No GPU. Runs in seconds.

## 00 — Setup

In [ ]:
import os, json, pickle, glob, time
import numpy as np
print("numpy:", np.__version__)

## 01 — Write conformal.py (fixed version, baked in)

In [ ]:
%%writefile conformal.py
from __future__ import annotations
from typing import Dict, List, Optional, Tuple
import numpy as np

def empirical_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    if n == 0:
        return float("inf")
    level = np.ceil((n + 1) * (1 - alpha)) / n
    level = min(level, 1.0)
    return float(np.quantile(scores, level, method="higher"))

def pb_count(scores: np.ndarray, probs: np.ndarray) -> np.ndarray:
    return (scores[:, None] * probs).sum(axis=0)

def pb_variance(scores: np.ndarray, probs: np.ndarray) -> np.ndarray:
    w = scores[:, None] * probs
    return (w * (1.0 - w)).sum(axis=0)

def pb_covariance(scores: np.ndarray, probs: np.ndarray) -> np.ndarray:
    K = probs.shape[1]
    cov = np.zeros((K, K))
    for j in range(K):
        for k in range(K):
            delta = 1.0 if j == k else 0.0
            cov[j, k] = (scores * probs[:, j] * (delta - probs[:, k])).sum()
    return cov

class MarginalSplitConformal:
    def __init__(self, alpha: float = 0.1):
        self.alpha = alpha
        self.q_per_class: Optional[np.ndarray] = None

    def fit(self, cal_preds: List[Dict], cal_gt: List[np.ndarray]) -> "MarginalSplitConformal":
        K = cal_gt[0].shape[0]
        scores_per_class = [[] for _ in range(K)]

        for pred, gt in zip(cal_preds, cal_gt):
            s = pred["scores"]
            p = pred["probs"]
            if len(s) == 0:
                
                for k in range(K):
                    scores_per_class[k].append(float(gt[k]))
                continue
            n_pred = pb_count(s, p)
            sigma = np.sqrt(pb_variance(s, p) + 1e-6)
            for k in range(K):
                err = abs(gt[k] - n_pred[k]) / sigma[k]
                scores_per_class[k].append(err)

        self.q_per_class = np.array([
            empirical_quantile(np.array(scores_per_class[k]), self.alpha)
            for k in range(K)
        ])
        return self

    def predict_interval(self, pred: Dict) -> Tuple[np.ndarray, np.ndarray]:
        K = len(self.q_per_class)
        s = pred["scores"]
        p = pred["probs"]
        if len(s) == 0:
            return np.zeros(K), np.zeros(K)
        n_pred = pb_count(s, p)
        sigma = np.sqrt(pb_variance(s, p) + 1e-6)
        lower = np.maximum(0, n_pred - self.q_per_class * sigma)
        upper = n_pred + self.q_per_class * sigma
        return lower, upper

class AdaptiveConformalInference:
    def __init__(self, alpha_target: float = 0.1, gamma: float = 0.05):
        self.alpha_target = alpha_target
        self.gamma = gamma
        self.alpha_t = alpha_target
        self.history_q: List[float] = []
        self.history_scores: List[float] = []  

    def reset(self):
        self.alpha_t = self.alpha_target
        self.history_scores = []

    def update(self, score_t: float, covered_t: bool):
        self.history_scores.append(score_t)
        err_t = 0.0 if covered_t else 1.0
        self.alpha_t = self.alpha_t + self.gamma * (self.alpha_target - err_t)
        
        self.alpha_t = max(1e-3, min(0.5, self.alpha_t))

    def get_quantile(self) -> float:
        if not self.history_scores:
            return 1.0
        return empirical_quantile(np.array(self.history_scores), self.alpha_t)

class ShiftAwareACI(AdaptiveConformalInference):
    def __init__(self, alpha_target: float = 0.1, gamma_0: float = 0.05,
                 lambda_: float = 3.0, gamma_max: float = 0.15):
        super().__init__(alpha_target, gamma_0)
        self.gamma_0 = gamma_0
        self.lambda_ = lambda_
        self.gamma_max = gamma_max
        self.gamma_t_last = gamma_0

    def update(self, score_t: float, covered_t: bool, delta_t: float = 0.0):
        self.history_scores.append(score_t)
        gamma_t = self.gamma_0 * (1.0 + self.lambda_ * max(0.0, delta_t))
        gamma_t = min(gamma_t, self.gamma_max)
        self.gamma_t_last = gamma_t
        err_t = 0.0 if covered_t else 1.0
        self.alpha_t = self.alpha_t + gamma_t * (self.alpha_target - err_t)
        self.alpha_t = max(1e-3, min(0.5, self.alpha_t))

class RollingShiftDetector:
    def __init__(self, window: int = 100, cap: float = 1.0):
        self.window = window
        self.cap = cap
        self.baseline: Optional[float] = None
        self.recent: List[float] = []

    def fit_baseline(self, cal_scores) -> "RollingShiftDetector":
        self.baseline = float(np.median(np.asarray(cal_scores))) + 1e-6
        return self

    def step(self, score_t: float) -> float:
        self.recent.append(float(score_t))
        if len(self.recent) > self.window:
            self.recent.pop(0)
        cur = float(np.median(self.recent))
        delta = (cur - self.baseline) / self.baseline
        return float(np.clip(delta, 0.0, self.cap))

class PBAwareJointConformalOnline:
    def __init__(self, alpha: float = 0.1, window: int = 300):
        self.alpha = alpha
        self.window = window
        self.scores: List[float] = []

    def warmstart(self, cal_scores) -> "PBAwareJointConformalOnline":
        self.scores = list(np.asarray(cal_scores)[-self.window:])
        return self

    def get_quantile(self) -> float:
        if not self.scores:
            return float("inf")
        return empirical_quantile(np.asarray(self.scores[-self.window:]), self.alpha)

    def update(self, score_t: float):
        self.scores.append(float(score_t))
        if len(self.scores) > self.window:
            self.scores = self.scores[-self.window:]

def local_coverage_stats(covered_list, window: int = 100) -> Dict[str, float]:
    c = np.asarray(covered_list, dtype=float)
    n = len(c)
    if n == 0:
        return {"min_local_cov": 0.0, "max_miss_run": 0}
    if n >= window:
        roll = np.convolve(c, np.ones(window) / window, mode="valid")
        min_local = float(roll.min())
    else:
        min_local = float(c.mean())
    run = mx = 0
    for v in covered_list:
        run = 0 if v else run + 1
        mx = max(mx, run)
    return {"min_local_cov": min_local, "max_miss_run": int(mx)}

class PBAwareJointConformal:
    def __init__(self, alpha: float = 0.1):
        self.alpha = alpha
        self.q: float = 0.0

    def fit(self, cal_preds: List[Dict], cal_gt: List[np.ndarray]) -> "PBAwareJointConformal":
        scores = []
        for pred, gt in zip(cal_preds, cal_gt):
            s = pred["scores"]
            p = pred["probs"]
            K = len(gt)
            if len(s) == 0:
                
                sigma_eps = 1.0
                S_t = max(abs(gt[k]) / sigma_eps for k in range(K))
            else:
                n_pred = pb_count(s, p)
                sigma = np.sqrt(pb_variance(s, p) + 1e-6)
                S_t = max(abs(gt[k] - n_pred[k]) / sigma[k] for k in range(K))
            scores.append(S_t)
        self.q = empirical_quantile(np.array(scores), self.alpha)
        return self

    def predict_interval(self, pred: Dict) -> Tuple[np.ndarray, np.ndarray]:
        s = pred["scores"]
        p = pred["probs"]
        K = pred.get("K", 5)
        if len(s) == 0:
            return np.zeros(K), np.zeros(K)
        n_pred = pb_count(s, p)
        sigma = np.sqrt(pb_variance(s, p) + 1e-6)
        lower = np.maximum(0, n_pred - self.q * sigma)
        upper = n_pred + self.q * sigma
        return lower, upper

class ClassStratifiedConformal:
    def __init__(self, alpha: float = 0.1, bonferroni: bool = True):
        self.alpha = alpha
        self.bonferroni = bonferroni
        self.q_per_class: Optional[np.ndarray] = None

    def fit(self, cal_preds: List[Dict], cal_gt: List[np.ndarray]) -> "ClassStratifiedConformal":
        K = cal_gt[0].shape[0]
        alpha_eff = self.alpha / K if self.bonferroni else self.alpha
        scores_per_class = [[] for _ in range(K)]

        for pred, gt in zip(cal_preds, cal_gt):
            s = pred["scores"]
            p = pred["probs"]
            if len(s) == 0:
                continue
            n_pred = pb_count(s, p)
            sigma = np.sqrt(pb_variance(s, p) + 1e-6)
            for k in range(K):
                if gt[k] > 0:  
                    err = abs(gt[k] - n_pred[k]) / sigma[k]
                    scores_per_class[k].append(err)

        self.q_per_class = np.array([
            empirical_quantile(np.array(scores_per_class[k]) if scores_per_class[k]
                              else np.array([1.0]), alpha_eff)
            for k in range(K)
        ])
        return self

    def predict_interval(self, pred: Dict) -> Tuple[np.ndarray, np.ndarray]:
        K = len(self.q_per_class)
        s = pred["scores"]
        p = pred["probs"]
        if len(s) == 0:
            return np.zeros(K), np.zeros(K)
        n_pred = pb_count(s, p)
        sigma = np.sqrt(pb_variance(s, p) + 1e-6)
        lower = np.maximum(0, n_pred - self.q_per_class * sigma)
        upper = n_pred + self.q_per_class * sigma
        return lower, upper

def coverage_per_class(intervals_lo: np.ndarray, intervals_hi: np.ndarray,
                       gt_counts: np.ndarray) -> np.ndarray:
    covered = (gt_counts >= intervals_lo) & (gt_counts <= intervals_hi)
    return covered.mean(axis=0)

def joint_coverage(intervals_lo: np.ndarray, intervals_hi: np.ndarray,
                   gt_counts: np.ndarray) -> float:
    covered_all = ((gt_counts >= intervals_lo) & (gt_counts <= intervals_hi)).all(axis=1)
    return float(covered_all.mean())

def avg_width_per_class(intervals_lo: np.ndarray, intervals_hi: np.ndarray) -> np.ndarray:
    return (intervals_hi - intervals_lo).mean(axis=0)

def macro_width(intervals_lo: np.ndarray, intervals_hi: np.ndarray) -> float:
    return float(avg_width_per_class(intervals_lo, intervals_hi).mean())

def split_calibration_test(preds: List[Dict], gts: List[np.ndarray],
                           cal_ratio: float = 0.5,
                           seed: int = 42) -> Tuple[List, List, List, List]:
    n = len(preds)
    rng = np.random.RandomState(seed)
    indices = rng.permutation(n)
    n_cal = int(n * cal_ratio)
    cal_idx = indices[:n_cal]
    test_idx = indices[n_cal:]
    cal_preds = [preds[i] for i in cal_idx]
    cal_gt = [gts[i] for i in cal_idx]
    test_preds = [preds[i] for i in test_idx]
    test_gt = [gts[i] for i in test_idx]
    return cal_preds, cal_gt, test_preds, test_gt

In [ ]:
import sys
if "." not in sys.path:
    sys.path.insert(0, ".")
from conformal import (
    MarginalSplitConformal, AdaptiveConformalInference, ShiftAwareACI,
    PBAwareJointConformal, PBAwareJointConformalOnline, ClassStratifiedConformal,
    RollingShiftDetector, local_coverage_stats,
    coverage_per_class, joint_coverage, avg_width_per_class, macro_width,
    pb_count, pb_variance,
)
print("conformal helpers loaded.")

## 02 — Benchmark functions (same as main Phase C)

In [ ]:
ALPHA = 0.1
GAMMA_0 = 0.05
LAMBDA = 3.0
GAMMA_MAX = 0.15
DETECTOR_WINDOW = 100
PBJCI_WINDOW = 300
LOCAL_WINDOW = 100

EVAL_SETTINGS = ["in_dist", "mild_shift", "severe_shift", "temporal_drift"]
METHODS = ["marginal_split", "aci", "sa_aci", "pb_jci", "pb_jci_online", "class_strat"]

def get_nonconformity_scores(preds, gt_list):
    scores = []
    for p, gt in zip(preds, gt_list):
        if len(p["scores"]) == 0:
            scores.append(float(abs(gt).max()))
            continue
        n_p = pb_count(p["scores"], p["probs"])
        sigma = np.sqrt(pb_variance(p["scores"], p["probs"]) + 1e-6)
        S = max(abs(gt[k] - n_p[k]) / sigma[k] for k in range(5))
        scores.append(S)
    return np.array(scores)

def _interval(p, q, K=5):
    if len(p["scores"]) == 0:
        return np.zeros(K), np.zeros(K)
    n_p = pb_count(p["scores"], p["probs"])
    sigma = np.sqrt(pb_variance(p["scores"], p["probs"]) + 1e-6)
    return np.maximum(0, n_p - q * sigma), n_p + q * sigma

def _score_one(p, gt, K=5):
    if len(p["scores"]) == 0:
        return float(abs(gt).max())
    n_p = pb_count(p["scores"], p["probs"])
    sigma = np.sqrt(pb_variance(p["scores"], p["probs"]) + 1e-6)
    return max(abs(gt[k] - n_p[k]) / sigma[k] for k in range(K))

def _summary(los, his, covered_list, gt_arr, online=False):
    cov_pc = coverage_per_class(los, his, gt_arr)
    width = avg_width_per_class(los, his)
    jc = float(np.mean(covered_list)) if online else joint_coverage(los, his, gt_arr)
    loc = local_coverage_stats(covered_list, window=LOCAL_WINDOW)
    return {"cov_per_class": cov_pc.tolist(),
            "marginal_coverage": float(cov_pc.mean()),
            "joint_coverage": jc,
            "width_per_class": width.tolist(),
            "macro_width": float(width.mean()),
            **loc}

def eval_static_method(method, test_preds, test_gt):
    los, his = [], []
    for p in test_preds:
        p["K"] = 5
        lo, hi = method.predict_interval(p)
        los.append(lo); his.append(hi)
    los = np.array(los); his = np.array(his); gt_arr = np.array(test_gt)
    covered_list = ((gt_arr >= los) & (gt_arr <= his)).all(axis=1).tolist()
    return _summary(los, his, covered_list, gt_arr, online=False)

def eval_aci_method(method, test_preds, test_gt, cal_scores, detector=None):
    method.reset()
    method.history_scores = list(cal_scores)
    los_list, his_list, covered_list = [], [], []
    K = 5
    for p, gt in zip(test_preds, test_gt):
        q = method.get_quantile()
        lo, hi = _interval(p, q, K)
        los_list.append(lo); his_list.append(hi)
        covered = bool(((gt >= lo) & (gt <= hi)).all())
        covered_list.append(covered)
        S_t = _score_one(p, gt, K)
        if isinstance(method, ShiftAwareACI):
            delta = detector.step(S_t) if detector is not None else 0.0
            method.update(S_t, covered, delta_t=delta)
        else:
            method.update(S_t, covered)
    los = np.array(los_list); his = np.array(his_list); gt_arr = np.array(test_gt)
    return _summary(los, his, covered_list, gt_arr, online=True)

def eval_online_window(method, test_preds, test_gt, cal_scores):
    method.warmstart(cal_scores)
    los_list, his_list, covered_list = [], [], []
    K = 5
    for p, gt in zip(test_preds, test_gt):
        q = method.get_quantile()
        lo, hi = _interval(p, q, K)
        los_list.append(lo); his_list.append(hi)
        covered = bool(((gt >= lo) & (gt <= hi)).all())
        covered_list.append(covered)
        method.update(_score_one(p, gt, K))
    los = np.array(los_list); his = np.array(his_list); gt_arr = np.array(test_gt)
    return _summary(los, his, covered_list, gt_arr, online=True)

def make_split(n, cal_ratio, seed):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(n)
    n_cal = int(n * cal_ratio)
    return idx[:n_cal], idx[n_cal:]

def run_benchmark(cal_seed, verbose=False):
    n = len(gt_counts)
    cal_idx, test_idx = make_split(n, 0.5, cal_seed)

    cal_preds = [predictions_by_setting["in_dist"][i] for i in cal_idx]
    cal_gt    = [gt_counts[i] for i in cal_idx]
    test_gt   = [gt_counts[i] for i in test_idx]
    test_preds = {s: [predictions_by_setting[s][i] for i in test_idx] for s in SETTINGS}

    cal_scores = get_nonconformity_scores(cal_preds, cal_gt)

    third = len(test_idx) // 3
    drift_preds = (test_preds["in_dist"][:third]
                   + test_preds["mild_shift"][third:2 * third]
                   + test_preds["severe_shift"][2 * third:])
    eval_streams = {
        "in_dist":        (test_preds["in_dist"],      test_gt),
        "mild_shift":     (test_preds["mild_shift"],    test_gt),
        "severe_shift":   (test_preds["severe_shift"],  test_gt),
        "temporal_drift": (drift_preds,                 test_gt),
    }

    msc = MarginalSplitConformal(alpha=ALPHA).fit(cal_preds, cal_gt)
    pb_jci = PBAwareJointConformal(alpha=ALPHA).fit(cal_preds, cal_gt)
    csc = ClassStratifiedConformal(alpha=ALPHA, bonferroni=True).fit(cal_preds, cal_gt)

    res = {s: {} for s in eval_streams}
    for setting, (tp, tg) in eval_streams.items():
        res[setting]["marginal_split"] = eval_static_method(msc, tp, tg)
        res[setting]["pb_jci"] = eval_static_method(pb_jci, tp, tg)
        res[setting]["class_strat"] = eval_static_method(csc, tp, tg)

        aci = AdaptiveConformalInference(alpha_target=ALPHA, gamma=GAMMA_0)
        res[setting]["aci"] = eval_aci_method(aci, tp, tg, cal_scores)

        sa_aci = ShiftAwareACI(alpha_target=ALPHA, gamma_0=GAMMA_0,
                               lambda_=LAMBDA, gamma_max=GAMMA_MAX)
        detector = RollingShiftDetector(window=DETECTOR_WINDOW).fit_baseline(cal_scores)
        res[setting]["sa_aci"] = eval_aci_method(sa_aci, tp, tg, cal_scores, detector=detector)

        pbo = PBAwareJointConformalOnline(alpha=ALPHA, window=PBJCI_WINDOW)
        res[setting]["pb_jci_online"] = eval_online_window(pbo, tp, tg, cal_scores)

        if verbose:
            print(f"\\n=== {setting} ===")
            for m in METHODS:
                r = res[setting][m]
                print(f"  {m:16s}: marg={r['marginal_coverage']:.3f} "
                      f"joint={r['joint_coverage']:.3f} width={r['macro_width']:7.2f} "
                      f"minLocal={r['min_local_cov']:.3f} missRun={r['max_miss_run']}")
    return res, len(cal_idx), len(test_idx)

## 03 — Loop 3 model seeds x 5 cal seeds -> model-seed CI

`run_benchmark` reads globals `predictions_by_setting`, `gt_counts`, `SETTINGS`;
we reassign them per model-seed pkl, then aggregate across all 15 runs.

In [ ]:
WORK = "/kaggle/working"
MODEL_SEEDS = [42, 100, 200]
CAL_SEEDS   = [42, 100, 200, 300, 400]

def find_pkl(seed):
    for pat in [f"/kaggle/input/**/phase_C_preds_seed{seed}.pkl",
                f"/kaggle/input/**/phase_C_preds_seed{seed}_*.pkl"]:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    return None

method_names = {
    "marginal_split": "Marginal Split", "aci": "ACI (Gibbs-Candes)",
    "sa_aci": "SA-ACI (Ours)", "pb_jci": "PB-Aware JCI (Ours)",
    "pb_jci_online": "PB-JCI Online (Ours)", "class_strat": "Class-Strat Bonf",
}

raw = {s: {m: {"marginal_coverage": [], "joint_coverage": [], "macro_width": [],
               "min_local_cov": [], "max_miss_run": []}
           for m in METHODS} for s in EVAL_SETTINGS}
per_run = {}
n_test_last = None

for model_seed in MODEL_SEEDS:
    pkl = find_pkl(model_seed)
    assert pkl, f"pkl for model seed {model_seed} not found under /kaggle/input"
    with open(pkl, "rb") as f:
        d = pickle.load(f)
    # rebind globals that run_benchmark reads
    predictions_by_setting = d["predictions_by_setting"]
    gt_counts = np.asarray(d["gt_counts"])
    SETTINGS = list(predictions_by_setting.keys())
    globals().update(predictions_by_setting=predictions_by_setting,
                     gt_counts=gt_counts, SETTINGS=SETTINGS)
    print(f"model seed {model_seed}: {pkl}  N={len(gt_counts)}  settings={SETTINGS}")

    for cal_seed in CAL_SEEDS:
        res, _, n_test = run_benchmark(cal_seed, verbose=False)
        n_test_last = n_test
        per_run[f"model{model_seed}_cal{cal_seed}"] = res
        for s in EVAL_SETTINGS:
            for m in METHODS:
                for key in raw[s][m]:
                    raw[s][m][key].append(res[s][m][key])
    print(f"  done 5 cal seeds")

def ms(vals):
    a = np.asarray(vals, dtype=float)
    return float(a.mean()), float(a.std())

n_runs = len(MODEL_SEEDS) * len(CAL_SEEDS)
print("\n" + "=" * 120)
print(f"PHASE C MODEL-SEED CI | {len(MODEL_SEEDS)} model x {len(CAL_SEEDS)} cal = {n_runs} runs | N_test~{n_test_last}")
print("=" * 120)
print(f"\n{'Setting':<15s} | {'Method':<21s} | {'JointCov':>14s} | {'Width':>15s} | {'MinLocal':>14s}")
print("-" * 120)
agg = {s: {} for s in EVAL_SETTINGS}
for s in EVAL_SETTINGS:
    for m in METHODS:
        mc_m, mc_s = ms(raw[s][m]["marginal_coverage"])
        jc_m, jc_s = ms(raw[s][m]["joint_coverage"])
        w_m, w_s   = ms(raw[s][m]["macro_width"])
        ml_m, ml_s = ms(raw[s][m]["min_local_cov"])
        agg[s][m] = {"marg": [mc_m, mc_s], "joint": [jc_m, jc_s],
                     "width": [w_m, w_s], "min_local": [ml_m, ml_s]}
        print(f"{s:<15s} | {method_names[m]:<21s} | "
              f"{jc_m*100:>6.1f}+/-{jc_s*100:>4.1f}% | "
              f"{w_m:>8.2f}+/-{w_s:>5.2f} | "
              f"{ml_m*100:>6.1f}+/-{ml_s*100:>4.1f}%")
    print("-" * 120)

with open(f"{WORK}/phase_C_modelseed_results.json", "w") as f:
    json.dump({"config": {"model_seeds": MODEL_SEEDS, "cal_seeds": CAL_SEEDS,
                          "alpha": ALPHA, "lambda": LAMBDA, "gamma_max": GAMMA_MAX,
                          "n_runs": n_runs, "eval_settings": EVAL_SETTINGS,
                          "methods": METHODS},
               "per_run": per_run, "raw": raw, "aggregate": agg}, f, indent=2)
print(f"\nSaved: {WORK}/phase_C_modelseed_results.json")

## Notes

- Output `phase_C_modelseed_results.json` -> send to fill paper Table 4c.
- Same conformal logic as the main Phase C notebook; only difference is the
  loop over 3 model-seed pkls (model-seed + cal-seed CI combined).